# complete all plots

In [ ]:
# === 0. Setup =================================================================
options(warn = 1, scipen = 6, stringsAsFactors = FALSE)
t_start <- Sys.time()
log_msg <- function(msg) cat(sprintf("[%s] %s\n", format(Sys.time()), msg))
log_msg("=== ROSMAP PCC TDP-split pipeline starting ===")

In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(here)
  library(powerjoin)
  library(qs)
  library(DESeq2)
  library(ashr)
  library(broom)
  library(broom.mixed)
  library(BiocParallel)
  library(RhpcBLASctl)
  library(conflicted)
  library(ggrepel)
  library(ggbeeswarm)
  library(ggnewscale)
  library(ggh4x)
  library(ggrastr)
  library(emmeans)
  library(pheatmap)
})

In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(here)
  library(qs)
  library(DESeq2)
  library(ashr)
  library(broom)
  library(broom.mixed)
  library(BiocParallel)
  library(RhpcBLASctl)
  library(conflicted)
  library(ggrepel)
  library(ggbeeswarm)
  library(ggnewscale)
  library(ggh4x)
  library(ggrastr)
  library(emmeans)
  library(pheatmap)
  library(splines)
})

In [ ]:
conflicts_prefer(
  dplyr::count, dplyr::filter, dplyr::select, dplyr::summarise,
  dplyr::mutate, dplyr::rename, dplyr::lag, dplyr::first, dplyr::last,
  dplyr::between, dplyr::slice, dplyr::collapse, dplyr::desc,
  dplyr::union, dplyr::intersect, dplyr::setdiff,
  .quiet = TRUE
)

In [ ]:
library(caret)

In [ ]:
library(variancePartition)

In [ ]:
library(rentrez)

In [ ]:
library(xml2)

In [ ]:
#install.packages("glmmTMB")

In [ ]:
#install.packages("reformulas")

In [ ]:
library(reformulas)

In [ ]:
library(glmmTMB)

In [ ]:
sprint_all <- read_csv(
  "../../STAR_data/sprint_repeat_overlaps.csv",
  col_names = TRUE)

In [ ]:
dim(sprint_all)

In [ ]:
liu_repeat_counts <- qread("liu_repeat_counts_raw-fraction_AND_secondary_noMETA.qs")

In [ ]:
rm_raw <- read_csv(
  "repeatmasker_raw.csv",
  col_names = TRUE)

In [ ]:
liu_meta <- read_csv(
  "liu_sample_meta.csv",
  col_names = TRUE)

In [ ]:
repeat_whitelist <- rm_raw %>%
  distinct(class_family) %>%
  filter(
    !str_detect(class_family, coll("?")),
    !str_detect(class_family, coll("Jockey")),
    !str_detect(class_family, coll("Tx1")),
    !str_detect(class_family, coll("Dong")),
    !str_detect(class_family, coll("Penelope")),
    !str_detect(class_family, coll("5S-Deu-L2")),
    !str_detect(class_family, coll("tRNA-Deu")),
    !str_detect(class_family, coll("tRNA-RTE")),
    !str_detect(class_family, coll("RTE-BovB")),
    !str_detect(class_family, coll("Gypsy")),
    !str_detect(class_family, coll("RTE-X")),
    !str_detect(class_family, coll("CR1")),
    str_starts(class_family, coll("LTR")) |
      str_starts(class_family, coll("SINE")) |
      str_starts(class_family, coll("LINE")) |
      str_starts(class_family, coll("Retroposon/SVA")) #| 
      #str_starts(class_family, coll("Simple_repeat"))
  )


In [ ]:
sprint_all_agg <- sprint_all %>% 
  filter(type %in% c("AG", "TC")) %>% 
  mutate(
    nedited  = ad,
    unedited = dp - ad,             # true unedited depth
    sample_id = run_id
  ) %>% 
  group_by(class_family, sample_id) %>% 
  summarise(
    supporting_reads = sum(supporting_reads),
    nedited  = sum(nedited),
    unedited = sum(unedited),
    total    = nedited + unedited,  # now equals Σdp
    .groups  = "drop"
  )


In [ ]:
sprint_all_counts <- count(sprint_all, `class_family`)

In [ ]:
count_type_human_readable <- c(
  nedited_adjusted = "Edited reads",
  unedited_adjusted = "Unedited reads",
  total_adjusted = "Total reads",
  supporting_reads_adjusted = "Score adjusted"
)

In [ ]:
patient_id_color_mapping <- liu_meta %>%
  filter(
    tdp_43_status == "TDP-43 positive"
  ) %>%
  distinct(patient_sample_id) %>%
  arrange() %>%
  pull(patient_sample_id) %>% {
    set_names(
      paletteer::paletteer_d("colorblindr::OkabeIto", length(.)),
      .
    )
  }

In [ ]:
patient_id_color_mapping

In [ ]:
repeat_counts_mat <- liu_repeat_counts[["counts"]] %>%
  magrittr::set_colnames(
    str_remove(colnames(.), fixed(".sorted.bam"))
  ) %>% {
    .[, liu_meta$Run]
  }

In [ ]:
de_meta <- liu_meta %>%
  select(
    Run, patient_sample_id, tdp_43_status
  )

In [ ]:
repeat_counts_int <- round(repeat_counts_mat)

In [ ]:
de <- DESeqDataSetFromMatrix(
  repeat_counts_int,
  de_meta,
  design = ~ tdp_43_status + patient_sample_id
)

In [ ]:
liu_salmon <- qread("gse_126542_salmon_tximport.qs")

In [ ]:
liu_meta_raw <- read_csv("GSE126542.csv")

In [ ]:
liu_meta_filtered <- liu_meta_raw %>%
  filter(Run %in% colnames(liu_salmon$abundance)) %>%
  arrange(match(Run, colnames(liu_salmon$abundance)))

In [ ]:
sra_search <- entrez_search(db = "sra", term = "PRJNA522295")

In [ ]:
sra_runinfo <- entrez_fetch(db = "sra", id = sra_search$ids, rettype = "xml") %>%
  read_xml()

In [ ]:
sra_meta <- map(
  set_names(c("accession", "sample_name", "sample_title")),
  \(x) xml_attr(
    xml_find_all(sra_runinfo, "//Member"),
    x
  )
) %>%
  as_tibble() %>%
  distinct() %>%
  mutate(
    patient_sample_id = str_extract(sample_title, "S\\d+"),
    tdp_43_status = str_extract(sample_title, "/(TDP-43 (?:negative|positive)) ", group = 1)
  )

In [ ]:
liu_meta <- liu_meta_raw %>%
  power_inner_join(
    sra_meta,
    by = c("Sample Name" = "sample_name"),
    check = check_specs(
      unmatched_keys_left = "warn",
      duplicate_keys_right = "warn",
      duplicate_keys_left = "warn"
    )
  )

In [ ]:
liu_meta_fixed <- liu_meta %>%
  filter(Run %in% colnames(liu_salmon$counts)) %>%
  arrange(match(Run, colnames(liu_salmon$counts))) %>%
  column_to_rownames("Run")

In [ ]:
liu_de <- DESeq2::DESeqDataSetFromTximport(
  liu_salmon,
  liu_meta_fixed,
  design = ~1
) %>%
  DESeq2::estimateSizeFactors()

In [ ]:
liu_size_factors <- liu_de %>%
  DESeq2::normalizationFactors() %>%
  apply(2, \(x) exp(mean(log(x))))

In [ ]:
liu_meta_sf <- liu_meta %>%
  power_inner_join(
    enframe(
      liu_size_factors,
      name = "Run",
      value = "size_factor"
    ),
    by = "Run",
    check = check_specs(
      unmatched_keys_left = "warn",
      duplicate_keys_right = "warn",
      duplicate_keys_left = "warn",
      unmatched_keys_right = "warn"
    )
  )

In [ ]:
liu_meta_filtered <- liu_meta %>%
  filter(Run %in% names(liu_size_factors))

In [ ]:
liu_meta_sf <- liu_meta_filtered %>%
  power_inner_join(
    enframe(liu_size_factors, name = "Run", value = "size_factor"),
    by = "Run"
  )

In [ ]:
sizeFactors(de) <- liu_size_factors

In [ ]:
des <- DESeq(de)

In [ ]:
des <- qread(file.path("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam", "data", "liu_repeat_de.qs"))

In [ ]:
repeat_all_counts <- list(
  normalized = counts(des, normalized = TRUE),
  raw = counts(des, normalized = FALSE)
)

In [ ]:
repeat_norm_counts_family <- repeat_all_counts %>%
  map(\(x) as_tibble(x, rownames = "gene_id")) %>%
  bind_rows(.id = "normalized") %>%
  pivot_longer(
    -c(normalized, gene_id),
    names_to = "sample_id",
    values_to = "count"
  ) %>%
  pivot_wider(
    names_from = normalized,
    values_from = count
  ) %>%
  dplyr::rename(
    count = normalized,
    raw_count = raw
  ) %>%
  power_inner_join(
    rm_raw %>%
      transmute(
        gene_id = paste(rep_name, row_number(), sep = "_"),
        class_family,
        chromosome, start, end
      ),
    by = "gene_id",
    check = check_specs(
      unmatched_keys_left = "warn",
      duplicate_keys_right = "warn"
    )
  )

In [ ]:
head(repeat_norm_counts_family)

In [ ]:
repeat_norm_counts_family_agg <- repeat_norm_counts_family %>%
  group_by(
    sample_id, class_family
  ) %>%
  summarize(
    count = sum(count),
    raw_count = sum(raw_count),
    .groups = "drop"
  ) %>%
  semi_join(
    repeat_whitelist,
    by = "class_family"
  ) %>%
  power_inner_join(
    liu_meta %>%
      select(Run, patient_sample_id, tdp_43_status),
    by = c("sample_id" = "Run"),
    check = check_specs(
      unmatched_keys_left = "warn",
      unmatched_keys_right = "warn",
      duplicate_keys_right = "warn"
    )
  )

In [ ]:
repeat_norm_counts_family_agg_long <- repeat_norm_counts_family_agg %>%
  select(class_family, count, patient_sample_id, tdp_43_status) %>%
  pivot_wider(
    names_from = tdp_43_status,
    values_from = count
  ) %>%
  mutate(
    difference = `TDP-43 negative` - `TDP-43 positive`,
    lratio = log2(`TDP-43 negative` / `TDP-43 positive`)
  )

In [ ]:
de_agg_raw <- repeat_norm_counts_family_agg_long %>%
  group_by(
    class_family
  ) %>%
  summarize(
    res = list(
      lm(
        difference ~ 1,
        data = pick(everything())
      ) %>%
        broom::tidy()
    ),
    .groups = "drop"
  ) %>%
  unnest(res) %>%
  mutate(
    padj = p.adjust(p.value, method = "BH")
  )

In [ ]:
de_agg_raw_lratio <- repeat_norm_counts_family_agg_long %>%
  group_by(
    class_family
  ) %>%
  summarize(
    res = list(
      lm(
        lratio ~ 1,
        data = pick(everything())
      ) %>%
        broom::tidy()
    ),
    .groups = "drop"
  ) %>%
  unnest(res) %>%
  mutate(
    padj = p.adjust(p.value, method = "BH"),
    estimate_fc = 2^estimate
  )

In [ ]:
p <- de_agg_raw_lratio %>%
  # arrange(estimate) %>%
  mutate(
    across(
      class_family,
      \(x) fct_reorder(x, estimate)
    )
  ) %>%
  ggplot(
    aes(
      estimate,
      class_family,
      fill = padj < 0.05
    )
  ) +
  geom_col() +
  geom_text(
    aes(label = signif(padj, 2)),
    hjust = 1,
    nudge_x = -.001,
    color = "white"
  ) +
  scale_fill_manual(
    values = c(`TRUE` = "#c62a2a", `FALSE` = "#6d6868"),
    guide = "none"
  ) +
  labs(
    x = "log2 fold-change TDP-43 (+) vs (-)",
    y = NULL
  )

In [ ]:
p

In [ ]:
de_agg_raw_lratio

In [ ]:
extract_deseq_result <- function(de, contrast, name) {
  if (missing(contrast)) {
    res <- results(de, name = name)
  } else {
    res <- results(de, contrast = contrast)
  }
  shrunken <- lfcShrink(de, res = res, type = "ashr")
  shrunken %>%
    as_tibble(rownames = "gene_id") %>%
    left_join(
      res %>%
        as_tibble(rownames = "gene_id") %>%
        select(gene_id, log2FoldChange, lfcSE),
      by = "gene_id", suffix = c("", "_MLE")
    )
}

In [ ]:
resultsNames(des)
res_tdp43 <- extract_deseq_result(
  des,
  name = "tdp_43_status_TDP.43.positive_vs_TDP.43.negative"
)

In [ ]:
# Example using Box Plot + Jitter + Lines (can simplify)
library(ggplot2)
library(dplyr)
library(ggh4x)

# Assumes data prep (family_order, de_agg_raw_lratio for annotation) is done

# It is best practice to prepare a separate data frame for annotations,
# especially when using free scales in facets.
# This ensures annotations are placed correctly in each panel.
annotation_data <- repeat_norm_counts_family_agg %>%
  group_by(class_family) %>%
  # Find the maximum y-value in each facet to position the text above the data
  summarise(max_y = max(count + 1, na.rm = TRUE)) %>%
  ungroup() %>%
  # Join with the differential expression results to get p-values
  left_join(de_agg_raw_lratio, by = "class_family") %>%
  # Remove rows where a p-value couldn't be joined
  filter(!is.na(padj)) %>%
  
  # --- CHANGE IS HERE ---
  # Create a formatted label for the raw p-value instead of asterisks.
  # `signif(padj, 2)` will round the p-value to two significant digits.
  mutate(
    p_label = paste("p =", signif(padj, 2))
  )

In [ ]:
family_order <- c("SINE/Alu", "SINE/MIR", "Retroposon/SVA", "LINE/L1", "LINE/L2",
                "LTR", "LTR/ERV1", "LTR/ERVK", "LTR/ERVL", "LTR/ERVL-MaLR"
                )

In [ ]:
df_reordered <- repeat_norm_counts_family_agg %>%
  # Use mutate() to modify the 'class_family' column
  mutate(
    # Convert 'class_family' to a factor and set the levels to your custom order.
    # Any values in 'class_family' not present in 'family_order' will become NA.
    class_family = factor(class_family, levels = family_order)
  ) %>%
  # This is the crucial step: arrange the entire data frame by the new factor levels.
  arrange(class_family)

In [ ]:
df_reordered <- df_reordered %>%
  mutate(
    # Use case_when() to specify the replacement rules for the 'tdp_43_status' column.
    # It checks each condition in order and applies the new value.
    tdp_43_status = case_when(
      # When the value is "TDP-43 positive", change it to "Control".
      tdp_43_status == "TDP-43 positive" ~ "Control",
      
      # When the value is "TDP-43 negative", change it to "TDP-43 pathology".
      tdp_43_status == "TDP-43 negative" ~ "TDP-43 pathology",
      
      # The 'TRUE ~' part is a fallback that keeps any other value as it is.
      # This is good practice to prevent accidentally creating NA values if there
      # were other statuses you forgot about.
      TRUE ~ tdp_43_status
    )
  )

In [ ]:
annotation_data_reordered <- annotation_data %>%
  # Use mutate() to modify the 'class_family' column
  mutate(
    # Convert 'class_family' to a factor and set the levels to your custom order.
    # Any values in 'class_family' not present in 'family_order' will become NA.
    class_family = factor(class_family, levels = family_order)
  ) %>%
  # This is the crucial step: arrange the entire data frame by the new factor levels.
  arrange(class_family)

In [ ]:
# Load necessary libraries
library(ggplot2)
library(dplyr)
library(ggh4x)
library(ggtext) # Required for formatted text in facet labels

# --- Data Preparation ---
# This section assumes 'df_reordered' and 'de_agg_raw_lratio' are prepared.

# Combine p-value information with the main plotting data frame
df_plot_ready <- df_reordered %>%
  # Join the p-value information
  left_join(annotation_data_reordered, by = "class_family") %>%
  # Ensure the data is still ordered correctly by the `class_family` factor
  mutate(
    class_family = factor(class_family, levels = family_order),
    # --- CHANGES ARE HERE ---
    # 1. Create a column for significance stars
    significance_star = if_else(padj < 0.05, "  **", ""),
    
    # 2. Create a column for the color name
    p_color = if_else(padj < 0.05, "red", "black"),
    
    # 3. Create a formatted p-value number
    p_value_formatted = signif(padj, 2),
    
    # 4. Create the final facet label with detailed HTML formatting
    #facet_label = paste0(
    #  class_family, 
    #  "<br>p = <span style='color:", p_color, ";'>", 
    #  p_value_formatted, significance_star, 
    #  "</span>"
    #)
    facet_label = paste0(
      # Wrap the family name in a span to make it larger
      "<span style='font-size:11pt;'>", class_family, "</span>", 
      # Add two line breaks to create a gap
      "<br><br>", 
      # The "p =" part remains in the default (black) color
      "p = ", 
      # The p-value number and star get the conditional color
      "<span style='color:", p_color, ";'>", 
      p_value_formatted, significance_star, 
      "</span>"
    )
  ) %>%
  # Convert the new combined label to a factor to maintain the correct plotting order
  mutate(facet_label = factor(facet_label, levels = unique(facet_label)))

In [ ]:
df_plot_ready <- df_plot_ready[df_plot_ready$class_family %in% family_order, ]

In [ ]:
# --- Plotting Code ---
paired_boxplot_final <- df_plot_ready %>%
  ggplot(aes(x = tdp_43_status, y = count, fill = tdp_43_status)) +
  
  geom_line(aes(group = patient_sample_id), color = "grey50", alpha = 0.6, linewidth = 0.4) +
  geom_boxplot(outlier.shape = NA, alpha = 0.8, width = 0.7) +
  geom_point(aes(fill = tdp_43_status), # Color points by group
             position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.7),
             size = 2, alpha = 0.8, shape = 21) +
  
  # Manually set the fill colors using the darker shades
  scale_fill_manual(
    values = c("TDP-43 pathology" = "#7543A7", "Control" = "#FFB703"),
    guide = "none"
  ) +
  
  # Use a log scale for the y-axis
  scale_y_log10(
    labels = scales::label_scientific(digits = 2),
    expand = expansion(mult = c(0.05, 0.1)) # Reduced top expansion
  ) +
  
  # Facet by the new formatted label column
  ggh4x::facet_wrap2(~ facet_label, scales = "free_y", ncol = 6, axes = "y") +
                     
  # Apply a clean theme
  theme_bw(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    
    # Use element_markdown() to render the HTML in the facet label
    strip.text = element_markdown(face = "bold", size = 9, lineheight = 1.2),
    
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank()
  ) +
  
  # Add informative labels
  labs(
    title = "Repeat-Family Counts by TDP-43 Status \n",
    x = NULL,
    y = "Aggregated Read Count (log scale)"
  )

print(paired_boxplot_final)

In [ ]:
library(ggplot2); library(dplyr); library(ggh4x); library(ggtext)

# group order on the x-axis — this flips which colour sits left vs right
grp_levels <- c("TDP-43 pathology", "Control")   # purple first; swap the two strings to reverse

# --- data prep: clean strip label (family name only), flipped group order ----
df_plot_ready <- df_reordered %>%
  left_join(annotation_data_reordered, by = "class_family") %>%
  mutate(
    class_family  = factor(class_family,  levels = family_order),
    tdp_43_status = factor(tdp_43_status, levels = grp_levels)
  )
df_plot_ready <- df_plot_ready[df_plot_ready$class_family %in% family_order, ]
df_plot_ready <- df_plot_ready %>%
  mutate(tdp_43_status = fct_recode(tdp_43_status, "No TDP-43 pathology" = "Control"))
# --- per-family significance + bracket positions (only significant families) --
facet_stats <- df_plot_ready %>%
  group_by(class_family) %>%
  summarise(padj = dplyr::first(padj),
            dmax = max(count, na.rm = TRUE), .groups = "drop") %>%
  mutate(
    stars = case_when(padj < 0.01 ~ "**",
                      padj < 0.05 ~ "*",
                      TRUE        ~ NA_character_),
    y_tip  = dmax * 1.05,
    y_bar  = dmax * 1.12,
    y_star = dmax * 1.22
  ) %>%
  filter(!is.na(stars))                       # non-significant families get no bracket

# --- plot --------------------------------------------------------------------
paired_boxplot_final <- df_plot_ready %>%
  ggplot(aes(x = tdp_43_status, y = count, fill = tdp_43_status)) +
  geom_line(aes(group = patient_sample_id), color = "grey50", alpha = 0.6, linewidth = 0.4) +
  geom_boxplot(outlier.shape = NA, alpha = 0.8, width = 0.7) +
  geom_point(aes(fill = tdp_43_status),
             position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.7),
             size = 2, alpha = 0.8, shape = 21) +
  # significance bracket + star (significant families only)
  geom_segment(data = facet_stats, inherit.aes = FALSE, linewidth = 0.6, colour = "grey20",
               aes(x = 1, xend = 2, y = y_bar, yend = y_bar)) +
  geom_segment(data = facet_stats, inherit.aes = FALSE, linewidth = 0.6, colour = "grey20",
               aes(x = 1, xend = 1, y = y_tip, yend = y_bar)) +
  geom_segment(data = facet_stats, inherit.aes = FALSE, linewidth = 0.6, colour = "grey20",
               aes(x = 2, xend = 2, y = y_tip, yend = y_bar)) +
  geom_text(data = facet_stats, inherit.aes = FALSE,
            aes(x = 1.5, y = y_star, label = stars),
            size = 5, fontface = "bold", colour = "grey20") +
  scale_fill_manual(values = c("TDP-43 pathology" = "#7543A7", "No TDP-43 pathology" = "#FFB703"),
                    guide = "none") +
  scale_y_log10(labels = scales::label_scientific(digits = 2),
                expand = expansion(mult = c(0.05, 0.18))) +   # extra top room for the bracket
  ggh4x::facet_wrap2(~ class_family, scales = "free_y", ncol = 5, axes = "y") +
  theme_bw(base_size = 12) +
  theme(
    plot.title  = element_text(face = "bold", size = 16, hjust = 0.5),
    strip.text  = element_markdown(face = "bold", size = 11, lineheight = 1.2),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
    panel.grid.minor   = element_blank(),
    panel.grid.major.x = element_blank()
  ) +
  labs(title = "Repeat-Family Counts by TDP-43 Status",
       x = NULL, y = "Normalized Aggregated Read Count (log scale)",
       caption = "* p < 0.05    ** p < 0.01")
print(paired_boxplot_final)

In [ ]:
library(ggplot2); library(dplyr); library(ggh4x); library(ggtext); library(forcats)

grp_levels <- c("TDP-43 pathology", "Control")

df_plot_ready <- df_reordered %>%
  left_join(annotation_data_reordered, by = "class_family") %>%
  mutate(class_family  = factor(class_family,  levels = family_order),
         tdp_43_status = factor(tdp_43_status, levels = grp_levels))
df_plot_ready <- df_plot_ready[df_plot_ready$class_family %in% family_order, ]
df_plot_ready <- df_plot_ready %>%
  mutate(tdp_43_status = fct_recode(tdp_43_status, "No TDP-43 pathology" = "Control"))

# --- per-family significance + bracket positions (significant families only) --
facet_stats <- df_plot_ready %>%
  group_by(class_family) %>%
  summarise(padj = dplyr::first(padj),
            dmax = max(count, na.rm = TRUE), .groups = "drop") %>%
  mutate(
    stars = case_when(padj < 0.01 ~ "**",
                      padj < 0.05 ~ "*",
                      TRUE        ~ NA_character_),
    y_tip  = dmax * 1.03,    # lowered to match the tick headroom
    y_bar  = dmax * 1.08,
    y_star = dmax * 1.15
  ) %>%
  filter(!is.na(stars))

# --- even, evenly-log-spaced y ticks: 4 per panel ----------------------------
even_log_breaks <- function(n = 4, top_inset = 0.15, bottom_inset = 0.02) {
  function(limits) {
    lo <- log10(min(limits)); hi <- log10(max(limits))
    R  <- hi - lo
    10^seq(lo + bottom_inset * R, hi - top_inset * R, length.out = n)
  }
}

# --- plot --------------------------------------------------------------------
paired_boxplot_final <- df_plot_ready %>%
  ggplot(aes(x = tdp_43_status, y = count, fill = tdp_43_status)) +
  geom_line(aes(group = patient_sample_id), color = "grey50", alpha = 0.6, linewidth = 0.4) +
  geom_boxplot(outlier.shape = NA, alpha = 0.8, width = 0.7) +
  geom_point(aes(fill = tdp_43_status),
             position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.7),
             size = 2, alpha = 0.8, shape = 21) +
  geom_segment(data = facet_stats, inherit.aes = FALSE, linewidth = 0.6, colour = "grey20",
               aes(x = 1, xend = 2, y = y_bar, yend = y_bar)) +
  geom_segment(data = facet_stats, inherit.aes = FALSE, linewidth = 0.6, colour = "grey20",
               aes(x = 1, xend = 1, y = y_tip, yend = y_bar)) +
  geom_segment(data = facet_stats, inherit.aes = FALSE, linewidth = 0.6, colour = "grey20",
               aes(x = 2, xend = 2, y = y_tip, yend = y_bar)) +
  geom_text(data = facet_stats, inherit.aes = FALSE,
            aes(x = 1.5, y = y_star, label = stars),
            size = 5, fontface = "bold", colour = "grey20") +
  scale_fill_manual(values = c("TDP-43 pathology" = "#7543A7", "No TDP-43 pathology" = "#FFB703"),
                    guide = "none") +
  scale_y_log10(
    breaks = even_log_breaks(n = 4),                       # <-- evenly-log-spaced ticks
    labels = scales::label_scientific(digits = 2),
    expand = expansion(mult = c(0.05, 0.07))               # <-- was 0.18; bracket is lower now
  ) +
  ggh4x::facet_wrap2(~ class_family, scales = "free_y", ncol = 5, axes = "y") +
  theme_bw(base_size = 12) +
  theme(
    plot.title  = element_text(face = "bold", size = 16, hjust = 0.5),
    strip.text  = element_markdown(face = "bold", size = 11, lineheight = 1.2),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
    axis.text.y = element_text(size = 9),
    panel.grid.minor   = element_blank(),
    panel.grid.major.x = element_blank()
  ) +
  labs(title = "Repeat-Family Counts by TDP-43 Status",
       x = NULL, y = "Normalized Aggregated Read Count (log scale)",
       caption = "* p < 0.05    ** p < 0.01")
print(paired_boxplot_final)

In [ ]:
stem <- here::here("results_rosmap_pcc_tdp_split-final", "plots", "te_paired_boxplots_by_family")

ggsave(paste0(stem, ".pdf"), paired_boxplot_final, width = 14, height = 10, device = cairo_pdf, bg = "white")
ggsave(paste0(stem, ".svg"), paired_boxplot_final, width = 14, height = 10, bg = "white")
ggsave(paste0(stem, ".png"), paired_boxplot_final, width = 14, height = 10, dpi = 600, bg = "white")

In [ ]:
dim(res_tdp43)

In [ ]:
head(res_tdp43)

In [ ]:
tail(res_tdp43)

In [ ]:
rm_raw_2 <- rm_raw[rm_raw$class_family %in% c('SINE/Alu', 'SINE/MIR', 'Retroposon/SVA','LINE/L1','LINE/L2','LTR','LTR/ERV1','LTR/ERVK','LTR/ERVL','LTR/ERVL-MaLR'), ]

In [ ]:
qsave(res_tdp43, file.path("results_rosmap_pcc_tdp_split-final", "res_tdp43-liu-et-al.qs"),
      nthreads = 8, preset = "fast")

In [ ]:
res_tdp43 <- qread(file.path("results_rosmap_pcc_tdp_split-final", "res_tdp43-liu-et-al.qs"))  
                   

In [ ]:
res_tdp43_family <- res_tdp43 %>%
  power_inner_join(
    rm_raw %>%
      transmute(
        gene_id = paste(rep_name, row_number(), sep = "_"),
        class_family,
        chromosome, start, end
      ),
    by = "gene_id",
    check = check_specs(
      unmatched_keys_left = "warn",
      duplicate_keys_left = "warn",
      duplicate_keys_right = "warn"
    )
  )

In [ ]:
p <- res_tdp43_family %>%
  semi_join(
    repeat_whitelist,
    by = "class_family"
  ) %>%
  arrange(desc(pvalue)) %>%
  ggplot(
    aes(
      log2FoldChange,
      -log10(pvalue),
      color = padj < 0.05
    )
  ) +
  ggrastr::rasterize(
    geom_point(shape = 16, alpha = .6),
    dpi = 200, dev = "ragg"
  ) +
  scale_color_manual(
    values = c(`TRUE` = "red", `FALSE` = "black")
  ) +
  coord_cartesian(
    xlim = c(-6, 5)
  ) +
  facet_wrap(
    vars(class_family)
  )

ggsave(
  file.path("results_rosmap_pcc_tdp_split-final", "plots", "repeat_deseq_tdp43_volcano_by_family_selectd.pdf"),
  p, width = 7, height = 5
)

In [ ]:
res_tdp43_family_stats <- res_tdp43_family %>%
  filter(
    class_family != ".",
    !str_ends(class_family, fixed("?"))
  ) %>%
  group_by(
    class_family
  ) %>%
  summarize(
    n = n(),
    n_de = sum(padj < .05, na.rm = TRUE),
    n_down = sum(log2FoldChange < 0 & padj < .05, na.rm = TRUE),
    n_up = sum(log2FoldChange > 0 & padj < .05, na.rm = TRUE),
    prop_de = n_de / n,
    prop_down = n_down / n,
    prop_up = n_up / n,
    .groups = "drop"
  )


In [ ]:
res_tdp43_family_ttest <- res_tdp43_family %>%
  filter(
    class_family != ".",
    !str_ends(class_family, fixed("?"))
  ) %>%
  group_by(
    class_family
  ) %>%
  summarize(
    res = t.test(
      log2FoldChange,
      mu = 0
    ) %>%
      broom::tidy() %>%
      list(),
    .groups = "drop"
  ) %>%
  unnest(res) %>%
  mutate(
    padj = p.adjust(p.value, method = "BH")
  )

In [ ]:
sprint_all_agg_normalized <- sprint_all_agg %>%
  power_inner_join(
    enframe(
      liu_size_factors,
      name = "sample_id",
      value = "size_factor"
    ),
    by = "sample_id",
    check = check_specs(
      unmatched_keys_left = "warn",
      duplicate_keys_right = "warn"
    )
  ) %>%
  mutate(
    across(
      c(nedited, unedited, total, supporting_reads),
      .fns = list(
        normalized = \(x) x / size_factor
      )
    )
  )

In [ ]:
liu_meta2 <- read_csv(
  "liu_sample_meta.csv",
  col_names = TRUE)

In [ ]:
abs_test_data <- sprint_all_agg %>%
  semi_join(
    repeat_whitelist,
    by = "class_family"
  ) %>%
  bind_rows(
    group_by(
      .,
      sample_id
    ) %>%
    summarise(
      across(c(nedited, unedited, total, supporting_reads), sum),
      .groups = "drop"
    ) %>%
    mutate(
      class_family = "all"
    )
  ) %>%
  power_inner_join(
    liu_meta2 %>%
      select(Run, patient_sample_id, tdp_43_status, size_factor),
    by = c("sample_id" = "Run"),
    check = check_specs(
      unmatched_keys_left = "warn",
      duplicate_keys_right = "warn"
    )
  ) %>%
  mutate(
    across(
      c(nedited, unedited, total, supporting_reads),
      .fns = list(
        adjusted = \(x) x / size_factor
      )
    )
  )

abs_test_data_long <- abs_test_data %>%
  select(class_family, nedited, unedited, total, supporting_reads, patient_sample_id, tdp_43_status, ends_with("adjusted")) %>%
  pivot_longer(
    cols = -c(class_family, patient_sample_id, tdp_43_status),
    names_to = "type",
    values_to = "value"
  )



In [ ]:
abs_test_paired_raw <- abs_test_data_long %>%
  semi_join(
    repeat_whitelist,
    by = "class_family"
  ) %>%
  mutate(
    across(value, log10)
  ) %>%
  pivot_wider(
    names_from = `tdp_43_status`,
    values_from = -c(`tdp_43_status`, class_family, patient_sample_id, type)
  ) %>%
  mutate(
    difference = `TDP-43 negative` - `TDP-43 positive`
  ) %>%
  group_by(
    class_family, type
  ) %>%
  summarise(
    res = list(
      possibly(lm)(
        difference ~ 1,
        data = pick(everything())
      )
    ),
    .groups = "drop"
  )

In [ ]:
abs_test_paired_res <- abs_test_paired_raw %>%
  mutate(
    across(res, \(x) map(x, tidy))
  ) %>%
  unnest(res) %>%
  group_by(
    type
  ) %>%
  mutate(
    padj = p.adjust(p.value, method = "BH")
  ) %>%
  ungroup()

In [ ]:
library(lmerTest)

In [ ]:
abs_test_random_raw <- abs_test_data_long %>%
  crossing(
    trans = c("log10", "linear")
  ) %>%
  mutate(
    value = if_else(trans == "log10", log10(value), value)
  ) %>%
  group_by(
    type, `class_family`, trans
  ) %>%
  summarise(
    res = list(
      possibly(lmerTest::lmer)(
        value ~ tdp_43_status + (1 | patient_sample_id),
        data = cur_data()
      )
    ),
    .groups = "drop"
  )

In [ ]:
abs_test_res <- abs_test_random_raw %>%
  mutate(
    across(res, \(x) map(x, tidy))
  ) %>%
  unnest(res)

In [ ]:
editing_index_df <- repeat_norm_counts_family_agg %>%
  semi_join(
    repeat_whitelist,
    by = "class_family"
  ) %>%
  power_inner_join(
    abs_test_data,
    by = c("sample_id", "class_family", "patient_sample_id", "tdp_43_status"),
    check = check_specs(
      unmatched_keys_left = "warn",
      unmatched_keys_right = "warn",
      duplicate_keys_right = "warn",
      duplicate_keys_left = "warn"
    )
  ) %>%
  mutate(
    editing_index = 1000 * nedited_adjusted / count
  )

In [ ]:
editing_index_paired_difference <- editing_index_df %>%
  transmute(
    class_family,
    patient_sample_id,
    tdp_43_status,
    count,
    nedited_adjusted,
    editing_index = editing_index
  ) %>%
  pivot_wider(
    names_from = tdp_43_status,
    values_from = c(editing_index, count, nedited_adjusted)
  ) %>%
  mutate(
    difference = `count_TDP-43 negative` - `count_TDP-43 positive`,
    log_diff = log2(`count_TDP-43 negative` / `count_TDP-43 positive`)
  )

editing_index_paired_res <- editing_index_paired_difference %>%
  drop_na(difference) %>%
  group_by(
    class_family
  ) %>%
  filter(
    n() == 7
  ) %>%
  summarize(
    res = list(
      possibly(lm)(
        log_diff ~ 1,
        data = pick(everything())
      ) %>%
        broom::tidy()
    ),
    .groups = "drop"
  ) %>%
  unnest(res) %>%
  mutate(
    padj = p.adjust(p.value, method = "BH")
  )

library(lme4)
editing_index_paired_glmer_res <- editing_index_df %>%
  mutate(
    across(c(count, nedited_adjusted), round)
  ) %>%
  group_by(
    class_family
  ) %>%
  summarize(
    res = list(
      possibly(glmer)(
        cbind(nedited_adjusted, count) ~ tdp_43_status + (1 | patient_sample_id),
        data = pick(everything()),
        family = binomial
      ) %>%
        broom::tidy()
    ),
    .groups = "drop"
  ) %>%
  unnest(res) %>%
  filter(term == "tdp_43_statusTDP-43 positive") %>%
  mutate(
    padj = p.adjust(p.value, method = "BH")
  )


library(glmmTMB)
editing_index_paired_nb_raw <- editing_index_df %>%
  # mutate(
  #   across(c(count, nedited_adjusted), round)
  # ) %>%
  mutate(
    tdp_43_status = factor(tdp_43_status, levels = c("TDP-43 positive", "TDP-43 negative"))
  ) %>%
  group_by(
    class_family
  ) %>%
  summarize(
    res = list(
      possibly(glmmTMB)(
        nedited ~ tdp_43_status + offset(log(raw_count / size_factor)) + (1 | patient_sample_id),
        data = pick(everything()),
        family = nbinom2
      )
    ),
    .groups = "drop"
  )

editing_index_paired_nb_res <- editing_index_paired_nb_raw %>%
  mutate(
    res = map(res, tidy)
  ) %>%
  unnest(res) %>%
  filter(term == "tdp_43_statusTDP-43 negative") %>%
  mutate(
    padj = p.adjust(p.value, method = "BH")
  )

In [ ]:
1

In [ ]:
## ── Libraries ───────────────────────────────────────────────────────
library(tidyverse)     # dplyr, ggplot2, etc.
library(ggbeeswarm)    # geom_quasirandom()
library(ggnewscale)    # new_scale_color()
library(ggh4x)         # facet_wrap2()
library(scales)        # log_breaks(), percent_format()

## ── 0 ▸ data checks ---------------------------------------------------
stopifnot(
  all(c("editing_index_df", "editing_index_paired_nb_res") %in% ls()),
  "class_family"      %in% names(editing_index_df),
  "editing_index"     %in% names(editing_index_df),
  "tdp_43_status"     %in% names(editing_index_df),
  "patient_sample_id" %in% names(editing_index_df)
)

In [ ]:
## ---------------------------------------------------------------------
## 1 ▸ factor levels & pseudo‑count for log‑scale ----------------------
status_levels <- unique(editing_index_df$tdp_43_status)
editing_index_df <- editing_index_df %>%
  mutate(
    tdp_43_status = factor(tdp_43_status, levels = status_levels),
    editing_index_pseudo = editing_index + 1e-4      # avoid log10(0)
  )

## ---------------------------------------------------------------------
## 2 ▸ build a colour palette if one isn’t supplied --------------------
if (!exists("patient_id_color_mapping")) {
  message("patient_id_color_mapping not found — generating a viridis palette.")
  id_levels <- sort(unique(editing_index_df$patient_sample_id))
  patient_id_color_mapping <- setNames(
    viridisLite::viridis(length(id_levels), option = "C"),
    id_levels
  )
}

## ---------------------------------------------------------------------
## 3 ▸ facet order  (robust to missing effect column) ------------------

# 3a. try to detect a pre‑existing effect column -----------------------
effect_candidates <- c("log2FoldChange", "log2fc", "lfc", "estimate", "lratio")
effect_col <- intersect(effect_candidates, names(editing_index_paired_nb_res))[1]

if (!is.na(effect_col)) {
  facet_order <- editing_index_paired_nb_res %>%
    mutate(effect_abs = abs(.data[[effect_col]])) %>%
    arrange(desc(effect_abs)) %>%
    pull(class_family)
  message("Facet order based on existing column: ", effect_col)

} else {
  # 3b. compute a median paired log2 FC on the fly ---------------------
  message("No effect column in editing_index_paired_nb_res; ",
          "computing median log2 fold‑change per class_family.")
  
  tmp_effect <- editing_index_df %>%
    group_by(class_family, tdp_43_status) %>%
    summarise(med = median(editing_index, na.rm = TRUE), .groups = "drop") %>%
    pivot_wider(names_from = tdp_43_status, values_from = med) %>%
    mutate(log2fc_tmp = log2((`TDP43+` + 1e-6) / (`TDP43-` + 1e-6)))
  
  facet_order <- tmp_effect %>%
    arrange(desc(abs(log2fc_tmp))) %>%
    pull(class_family)

  # merge back for optional p‑value labels if you like:
  editing_index_paired_nb_res <- editing_index_paired_nb_res %>%
    left_join(tmp_effect %>% select(class_family, log2fc_tmp),
              by = "class_family")
}

# apply the order
editing_index_df <- editing_index_df %>%
  mutate(class_family = factor(class_family, levels = facet_order))


## ---------------------------------------------------------------------
## 4 ▸ the plot ---------------------------------------------------------
p <- ggplot(editing_index_df,
            aes(x      = tdp_43_status,
                y      = editing_index_pseudo,
                group  = patient_sample_id,
                colour = patient_sample_id)) +
  geom_line(alpha = .4, linewidth = .6, show.legend = FALSE) +
  geom_quasirandom(width = .12, size = 1.6, show.legend = FALSE) +

  scale_colour_manual(values = patient_id_color_mapping, guide = "none") +
  ggnewscale::new_scale_color() +

  ## FDR labels per facet
  geom_text(
    data = editing_index_paired_nb_res %>% filter(!is.na(padj)),
    aes(label = signif(padj, 2),
        colour = padj < 0.05),
    x = -Inf, y = Inf, hjust = 0, vjust = 1,
    inherit.aes = FALSE, na.rm = TRUE
  ) +
  scale_colour_manual(values = c(`TRUE` = "firebrick",
                                 `FALSE` = "grey20"),
                      guide = "none") +

  scale_y_log10(
    breaks  = log_breaks(n = 5, base = 10),
    labels  = label_number(),
    expand  = expansion(mult = c(.08, .25)),
    name    = "Editing index (+1e‑4)"
  ) +

  ggh4x::facet_wrap2(~ class_family,
                     scales = "free_y",
                     axes   = "y",
                     ncol   = 4,
                     strip  = ggh4x::strip_vanilla(size = "constant")) +

  theme_bw(base_size = 11) +
  theme(
    strip.text       = element_text(face = "bold", size = 9),
    axis.text.x      = element_text(angle = 45, hjust = 1),
    panel.grid.minor = element_blank()
  ) +
  labs(
    x = NULL,
    title = "Editing index per repeat class (paired by patient)"
  )

print(p)

## ---------------------------------------------------------------------
## 5 ▸ export -----------------------------------------------------------
out_dir <- "results_rosmap_pcc_tdp_split-final/plots/"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

ggsave(file.path(out_dir, "editing_index_paired_nb2.pdf"),
       p, width = 7, height = 5)
ggsave(file.path(out_dir, "editing_index_paired_nb2.png"),
       p, width = 7, height = 5, dpi = 300)

In [ ]:
head(res_tdp43_family)

In [ ]:
head(res_tdp43_family_stats)

In [ ]:
head(res_tdp43_family)

In [ ]:
res_tdp43_family2 <- res_tdp43_family[res_tdp43_family$class_family %in% family_order,]

In [ ]:
res_tdp43_1 <- qread(file.path("results_rosmap_pcc_tdp_split-final", "res_tdp43-liu-et-al.qs"))  
                   
res_tdp43_family_1 <- res_tdp43_1 %>%
  power_inner_join(
    rm_raw %>%
      transmute(
        gene_id = paste(rep_name, row_number(), sep = "_"),
        class_family,
        chromosome, start, end
      ),
    by = "gene_id",
    check = check_specs(
      unmatched_keys_left = "warn",
      duplicate_keys_left = "warn",
      duplicate_keys_right = "warn"
    )
  )

In [ ]:
des_2 <- qread("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam/data/liu_repeat_de.qs")

In [ ]:
repeat_all_counts_2 <- list(
  normalized = counts(des_2, normalized = TRUE),
  raw = counts(des_2, normalized = FALSE)
)

In [ ]:
extract_deseq_result <- function(de, contrast, name) {
  if (missing(contrast)) {
    res <- results(de, name = name)
  } else {
    res <- results(de, contrast = contrast)
  }
  shrunken <- lfcShrink(de, res = res, type = "ashr")
  shrunken %>%
    as_tibble(rownames = "gene_id") %>%
    left_join(
      res %>%
        as_tibble(rownames = "gene_id") %>%
        select(gene_id, log2FoldChange, lfcSE),
      by = "gene_id", suffix = c("", "_MLE")
    )
}

In [ ]:
resultsNames(des_2)
res_tdp43_2 <- extract_deseq_result(
  des_2,
  name = "tdp_43_status_TDP.43.positive_vs_TDP.43.negative"
)

In [ ]:
res_tdp43_family_2 <- res_tdp43_2 %>%
  power_inner_join(
    rm_raw %>%
      transmute(
        gene_id = paste(rep_name, row_number(), sep = "_"),
        class_family,
        chromosome, start, end
      ),
    by = "gene_id",
    check = check_specs(
      unmatched_keys_left = "warn",
      duplicate_keys_left = "warn",
      duplicate_keys_right = "warn"
    )
  )

In [ ]:
dim(res_tdp43_family_1)

In [ ]:
dim(res_tdp43_family_2)

In [ ]:
head(res_tdp43_family_1)

In [ ]:
head(res_tdp43_family_2)

In [ ]:
res_tdp43_family2 %>%
  filter(!is.na(padj), padj < 0.05, log2FoldChange > 0) %>%   # significant & up in TDP+
  count(class_family, name = "n") %>%
  arrange(desc(n)) %>%
  mutate(
    pct          = n / sum(n) * 100,
    label        = sprintf("%s (%s, %.1f%%)", class_family, comma(n), pct),
    class_family = factor(class_family, levels = class_family)   # keep size order
  )

In [ ]:
res_tdp43_family2 %>%
  filter(!is.na(padj), padj < 0.05, log2FoldChange > 1) %>%   # significant & up in TDP+
  count(class_family, name = "n") %>%
  arrange(desc(n)) %>%
  mutate(
    pct          = n / sum(n) * 100,
    label        = sprintf("%s (%s, %.1f%%)", class_family, comma(n), pct),
    class_family = factor(class_family, levels = class_family)   # keep size order
  )

In [ ]:
# === fixed TE family → colour map (read off te_upregulated_donut.png) ===
te_palette <- c(
  "SINE/Alu"       = "#4C78A8",  # blue
  "LINE/L1"        = "#F58518",  # orange
  "SINE/MIR"       = "#54A24B",  # green
  "LINE/L2"        = "#E45756",  # red
  "LTR/ERVL-MaLR"  = "#9D7BD8",  # purple
  "LTR/ERV1"       = "#8C6D4F",  # brown
  "LTR/ERVL"       = "#E377C2",  # pink
  "LTR/ERVK"       = "#7F7F7F",  # grey
  "Retroposon/SVA" = "#BCBD22",  # olive
  "LTR"            = "#17BECF"   # cyan
)

# safety net: any family NOT in the map gets a spare colour (prevents grey "NA" slices)
fill_te <- function(fams, base = te_palette,
                    spares = c("#1F77B4","#AEC7E8","#FF9DA7","#9C755F","#BAB0AC","#D37295")) {
  miss <- setdiff(as.character(fams), names(base))
  if (length(miss)) base <- c(base, setNames(spares[seq_along(miss)], miss))
  base
}

In [ ]:
library(dplyr); library(ggplot2); library(scales)

up_counts <- res_tdp43_family2 %>%
  filter(!is.na(padj), padj < 0.05, log2FoldChange > 0) %>%   # significant & up in TDP+
  count(class_family, name = "n") %>%
  arrange(desc(n)) %>%
  mutate(
    pct          = n / sum(n) * 100,
    label        = sprintf("%s (%s, %.1f%%)", class_family, comma(n), pct),
    class_family = factor(class_family, levels = class_family)   # keep size order
  )

# distinct qualitative palette (extend if you have >12 families)
#pal_vals <- c("#4C78A8","#F58518","#54A24B","#E45756","#9D7BD8","#8C6D4F",
#              "#E377C2","#7F7F7F","#BCBD22","#17BECF","#1F77B4","#AEC7E8")
#pal <- setNames(pal_vals[seq_len(nrow(up_counts))], levels(up_counts$class_family))

p_donut <- ggplot(up_counts, aes(x = 2, y = n, fill = class_family)) +
  geom_col(width = 1, colour = "white", linewidth = 0.6) +
  coord_polar(theta = "y") +
  xlim(0.5, 2.5) +
  scale_fill_manual(
    values = fill_te(levels(up_counts$class_family)),
    labels = setNames(up_counts$label, as.character(up_counts$class_family)),
    name   = "Transposable Element Families (count, %share)"
  ) +
  theme_void(base_size = 13) +
  theme(legend.title = element_text(face = "bold"),
        legend.text  = element_text(size = 11),
        legend.key.size = unit(13, "pt"))
p_donut

In [ ]:
stem <- here::here("results_rosmap_pcc_tdp_split-final", "plots", "Liu_et_al_te_intercept_pos_donut")
ggsave(paste0(stem, ".pdf"), p_donut, width = 11, height = 7, device = cairo_pdf, bg = "white")
ggsave(paste0(stem, ".svg"), p_donut, width = 11, height = 7, bg = "white")
ggsave(paste0(stem, ".png"), p_donut, width = 11, height = 7, dpi = 600, bg = "white")

In [ ]:
head(res_tdp43_family2)

In [ ]:
intercept_df <- bind_rows(
  tibble(gene_id = rownames(des_2), intercept = coef(des_2)[, "Intercept"])
)

In [ ]:
res_sel <- res_tdp43_family2 %>%
  left_join(intercept_df, by = "gene_id") %>%
  filter(intercept > 0)

# confirm all 11 families survive the filter
res_sel %>% count(class_family, name = "n_loci") %>% arrange(desc(n_loci))
n_distinct(res_sel$class_family)        # should be 10

In [ ]:
library(dplyr); library(ggplot2); library(scales)

make_te_donut <- function(df, title = "Transposable Element Families (count, %share)") {
  d <- df %>%
    count(class_family, name = "n") %>%
    arrange(desc(n)) %>%
    mutate(pct          = n / sum(n) * 100,
           label        = sprintf("%s (%s, %.1f%%)", class_family, comma(n), pct),
           class_family = factor(class_family, levels = class_family))

  ggplot(d, aes(x = 2, y = n, fill = class_family)) +
    geom_col(width = 1, colour = "white", linewidth = 0.6) +
    coord_polar(theta = "y") +
    xlim(0.5, 2.5) +
    scale_fill_manual(
      values = fill_te(levels(d$class_family)),
      labels = setNames(d$label, as.character(d$class_family)),
      name   = title
    ) +
    theme_void(base_size = 13) +
    theme(legend.title = element_text(face = "bold"),
          legend.text  = element_text(size = 11),
          legend.key.size = unit(13, "pt"))
}

p_donut_sel <- make_te_donut(res_sel)
p_donut_sel

In [ ]:
stem <- here::here("results_rosmap_pcc_tdp_split-final", "plots", "Liu_et_al_baseline_te_intercept_pos_donut")
ggsave(paste0(stem, ".pdf"), p_donut_sel, width = 11, height = 7, device = cairo_pdf, bg = "white")
ggsave(paste0(stem, ".svg"), p_donut_sel, width = 11, height = 7, bg = "white")
ggsave(paste0(stem, ".png"), p_donut_sel, width = 11, height = 7, dpi = 600, bg = "white")